In [39]:
import pandas as pd
import numpy as np
import pyreadstat

### 1. Import Hearings Metadata (from GovInfo)

In [ ]:
import requests, time, os, dotenv
dotenv.load_dotenv()

BASE = "https://api.govinfo.gov"
KEY = os.getenv('govinfokey')

params = {'api_key' : KEY,
          'pageSize' : 1000,
          'offsetMark' : '*'}

In [ ]:
# Step 1: Collect Package IDs
def get_pkg_ids(start = "2020-01-01T00:00:00Z", 
                end = "2025-12-31T00:00:00Z"):
    ids = []
    url = f"{BASE}/collections/CHRG/{start}/{end}"
    first = True
    while url:
        if first:
            r = requests.get(url, 
                         params=params).json()
            first = False
        
    else:
        r = requests.get(url,
                         params={"api_key": KEY}).json()
        ids += [p["packageId"] for p in r.get("packages", [])]
        url = r.get("nextPage")
        time.sleep(0.5)
    return ids

In [ ]:
test = get_pkg_ids()

In [ ]:
# Step 2: Collect Metadata
def get_metadata(pkg_id):
    r = requests.get(f"{BASE}/packages/{pkg_id}/summary",
                     params = params).json()
    return r

### 2. Explore Data (Replication data from Ban et al 2025)

In [ ]:
df, meta = pyreadstat.read_dta('../data/dyad_b.dta')

In [ ]:
df.head()

In [ ]:
df['polappt'].value_counts(dropna=False)

In [ ]:
# Word count and Proportion of keywords in Hearings (with Political appointee vs. carrer bureaucrat witness)
df.groupby('polappt')[['word_count', 'prop_keyword']].mean()

In [ ]:
# Appointee vs. career bureaucrat under divided govt
df.groupby(['polappt', 'divided_house'])[['word_count', 'prop_keyword']].std()

In [ ]:
df.groupby(['polappt', 'divided_senate'])[['word_count', 'prop_keyword']].std()

In [ ]:
# Oversight hearing
oversight = df[df['oversight'] == 1]
oversight.groupby('polappt')[['word_count', 'prop_keyword']].mean()

In [ ]:
df.groupby(['polappt', 'divided_house'])['ascore'].mean()

## 2. Explore Data (from Congress.API)

In [ ]:
# Inital Setup
import numpy as np
import pandas as pd
import requests
import json
import dotenv
import os
import yaml
import llm
import pprint

from lxml import etree
from lxml import html
from bs4 import BeautifulSoup

In [ ]:
botname = 'targ'
version = '0.0'
email = 'kve5hd@virginia.edu'
useragent =f'{botname}/{version} ({email}) python-requests/{requests.__version__}'
headers = {'User-Agent':useragent}
headers

In [ ]:
root = "https://api.congress.gov//v3"
dotenv.load_dotenv()
congresskey = os.getenv('congresskey')

params = {'format': 'json',
          'api_key': congresskey}

In [ ]:
endpoint = f'/hearing'

r = requests.get(root + endpoint,
                 params=params,
                 headers=headers)
r

In [ ]:
myjson = r.json()

In [ ]:
myjson